# Build the 5-minute SPX feature set

This notebook creates the base 5-minute dataset from the minute-level S&P 500 file and prepares it for later feature merging and modeling.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "Data").exists() and (candidate / "Notebooks").exists():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
SPX_MINUTE_FILE = DATA_DIR / "^SP500.Last.txt"
ANNUAL_FACTOR = 252 * 78

print(f"Project root: {PROJECT_ROOT}")
print(f"SPX file: {SPX_MINUTE_FILE}")

                 bar5  m1_open  m1_high   m1_low  m1_close  m2_open  m2_high  \
1 2025-02-03 09:35:00  5953.77  5956.22  5951.08   5953.47  5953.44  5955.97   
2 2025-02-03 09:40:00  5957.18  5961.90  5956.63   5961.90  5961.80  5961.80   
3 2025-02-03 09:45:00  5956.54  5956.54  5952.72   5954.66  5953.14  5953.80   
4 2025-02-03 09:50:00  5944.88  5947.36  5944.68   5945.53  5945.30  5946.88   
5 2025-02-03 09:55:00  5935.98  5938.52  5934.61   5938.52  5938.13  5939.09   

    m2_low  m2_close  m3_open  ...  m3_close  m4_open  m4_high   m4_low  \
1  5951.84   5955.59  5955.55  ...   5952.93  5953.06  5953.38  5948.25   
2  5955.65   5957.41  5957.22  ...   5956.59  5956.70  5957.97  5954.30   
3  5948.52   5949.65  5949.62  ...   5944.28  5943.82  5948.65  5942.81   
4  5943.19   5944.81  5945.35  ...   5948.34  5948.07  5949.26  5940.58   
5  5935.02   5935.64  5935.65  ...   5943.03  5943.39  5947.67  5939.47   

   m4_close  m5_open  m5_high   m5_low  m5_close  target_ann_vol  
1

,bar5,m1_open,m1_high,m1_low,m1_close,m2_open,m2_high,m2_low,m2_close,m3_open,...,m3_close,m4_open,m4_high,m4_low,m4_close,m5_open,m5_high,m5_low,m5_close,target_ann_vol
1,2025-02-03 09:35:00,5953.77,5956.22,5951.08,5953.47,5953.44,5955.97,5951.84,5955.59,5955.55,...,5952.93,5953.06,5953.38,5948.25,5950.81,5950.79,5957.74,5949.47,5956.99,0.158433
2,2025-02-03 09:40:00,5957.18,5961.90,5956.63,5961.90,5961.80,5961.80,5955.65,5957.41,5957.22,...,5956.59,5956.70,5957.97,5954.30,5957.21,5957.15,5958.11,5953.15,5957.04,0.214807
3,2025-02-03 09:45:00,5956.54,5956.54,5952.72,5954.66,5953.14,5953.80,5948.52,5949.65,5949.62,...,5944.28,5943.82,5948.65,5942.81,5947.97,5948.12,5948.34,5943.94,5944.83,0.226859
4,2025-02-03 09:50:00,5944.88,5947.36,5944.68,5945.53,5945.30,5946.88,5943.19,5944.81,5945.35,...,5948.34,5948.07,5949.26,5940.58,5940.95,5940.82,5941.80,5935.51,5936.01,0.221982
5,2025-02-03 09:55:00,5935.98,5938.52,5934.61,5938.52,5938.13,5939.09,5935.02,5935.64,5935.65,...,5943.03,5943.39,5947.67,5939.47,5939.47,5939.40,5942.97,5939.33,5942.03,0.150569
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17888,2025-12-31 15:30:00,6854.12,6854.83,6853.10,6854.37,6854.36,6856.25,6852.95,6856.21,6856.34,...,6857.38,6857.43,6861.48,6857.43,6860.84,6860.87,6860.87,6859.09,6859.48,0.082056
17889,2025-12-31 15:35:00,6859.43,6860.51,6859.14,6860.40,6860.37,6862.11,6860.03,6862.11,6862.08,...,6862.60,6862.61,6864.18,6862.61,6863.51,6863.51,6863.58,6859.98,6860.15,0.037446
17890,2025-12-31 15:40:00,6860.30,6861.06,6859.84,6859.84,6859.74,6859.78,6858.14,6858.65,6858.60,...,6859.62,6859.48,6860.70,6859.28,6860.57,6860.53,6860.75,6859.73,6860.60,0.056473
17891,2025-12-31 15:45:00,6860.60,6861.68,6860.25,6861.49,6861.50,6861.96,6860.45,6860.45,6860.42,...,6858.85,6858.78,6858.99,6857.74,6857.94,6857.96,6858.13,6856.40,6856.40,0.044002


In [ ]:
# Output columns: bar5 + 5-minute OHLC + target_ann_vol
out_cols = (
    ["bar5"] +
    [f"m{i}_{x}" for i in range(1, 6) for x in ["open", "high", "low", "close"]] +
    ["target_ann_vol"]
)

spx = pd.read_csv(
    SPX_MINUTE_FILE,
    sep=";",
    header=None,
    names=["dt_raw", "open", "high", "low", "close", "volume"],
    usecols=["dt_raw", "open", "high", "low", "close"],
)

spx["dt"] = (
    pd.to_datetime(spx["dt_raw"], format="%Y%m%d %H%M%S", errors="coerce")
      .dt.tz_localize("UTC")
      .dt.tz_convert("America/New_York")
      .dt.tz_localize(None)
)

spx = spx[
    (spx["dt"].dt.time >= pd.Timestamp("09:30").time())
    & (spx["dt"].dt.time <= pd.Timestamp("15:59").time())
].copy()

spx = (
    spx.dropna(subset=["dt"]) 
       .assign(
           date=lambda x: x["dt"].dt.date,
           bar5=lambda x: x["dt"].dt.floor("5min"),
           minute_no=lambda x: x.groupby(["date", "bar5"]).cumcount() + 1,
       )
)

spx[["open", "high", "low", "close"]] = spx[["open", "high", "low", "close"]].apply(
    pd.to_numeric, errors="coerce"
)

spx = spx.dropna(subset=["close"]).sort_values("dt")
spx["ret_1m"] = spx.groupby("date")["close"].transform(lambda x: np.log(x / x.shift(1)))

target = (
    spx.groupby(["date", "bar5"])["ret_1m"]
       .apply(lambda r: np.sqrt(np.nansum(r**2)) * np.sqrt(ANNUAL_FACTOR))
       .rename("target_ann_vol")
)

ohlc = (
    spx[spx["minute_no"].between(1, 5)]
      .pivot(index=["date", "bar5"], columns="minute_no", values=["open", "high", "low", "close"])
)
# Shift the OHLC features to align with the target being predicted from prior-bar price action.
ohlc = ohlc.groupby(level="date").shift(1)
ohlc.columns = [f"m{m}_{v}" for v, m in ohlc.columns]

rv_5m = (
    ohlc.join(target)
         .reset_index()
         .drop(columns="date")
         .loc[:, out_cols]
         .dropna()
)

print(rv_5m.head())
print(f"\nRows: {len(rv_5m):,}")
print(f"Columns: {len(rv_5m.columns)}")

df5 = rv_5m.copy()
print(df5.head())

After dropping zero targets: 13807 samples
Merged: 13807 samples × 34 features


,prev_5m_log_return_1,prev_5m_log_return_2,prev_5m_log_return_3,prev_5m_log_return_4,prev_day_log_return,day_of_week,day_of_month,prev_5m_high_low_range,prev_atm_iv_30dte,prev_atm_iv_0dte,prev_total_option_volume,prev_total_option_oi
trade_dt,,,,,,,,,,,,
2025-02-05 09:00:00,0.000000,0.000000,0.000000,0.000000,0.007199,2.0,5.0,0.00,0.126715,0.010670,446468.0,2962783.0
2025-02-05 09:05:00,0.000000,0.000000,0.000000,0.000000,0.007199,2.0,5.0,0.00,0.138287,0.100552,24290.0,933779.0
2025-02-05 09:10:00,0.000000,0.000000,0.000000,0.000000,0.007199,2.0,5.0,0.00,0.137851,0.098565,26689.0,952416.0
2025-02-05 09:55:00,-0.001316,-0.000030,0.000158,0.000735,0.007199,2.0,5.0,8.38,0.131740,0.134747,116848.0,1676743.0
2025-02-05 10:00:00,-0.000645,-0.001316,-0.000030,0.000158,0.007199,2.0,5.0,7.45,0.132913,0.132816,144866.0,1765872.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 16:10:00,0.000000,-0.000501,-0.000724,-0.000366,-0.001377,2.0,31.0,0.00,0.134136,0.004514,1485639.0,3419476.0
2025-12-31 16:15:00,0.000000,0.000000,-0.000501,-0.000724,-0.001377,2.0,31.0,0.00,0.134264,0.004514,1591649.0,3424061.0
2025-12-31 16:20:00,0.000000,0.000000,0.000000,-0.000501,-0.001377,2.0,31.0,0.00,0.135224,0.004514,1645519.0,3421506.0


In [ ]:
# Merge in any additional engineered feature tables when available
feature_candidates = [
    DATA_DIR / "train_features_5m_clean.parquet",
    DATA_DIR / "processed_sets" / "winsorized_1_99.parquet",
]
feature_files = [p for p in feature_candidates if p.exists()]

if feature_files:
    features_raw = pd.concat([pd.read_parquet(p) for p in feature_files]).sort_index()
    if isinstance(features_raw.index, pd.Index):
        common_times = sorted(set(pd.to_datetime(features_raw.index)) & set(df5["bar5"]))
        features_aligned = features_raw.loc[pd.to_datetime(features_raw.index).isin(common_times)]
        df5_aligned = df5[[c for c in df5.columns if c not in features_raw.columns or c == "bar5"]].copy()
        df5_aligned = df5_aligned[df5_aligned["bar5"].isin(common_times)].set_index("bar5")
        df5 = df5_aligned.join(features_aligned, how="inner").dropna(axis=1).reset_index()
else:
    print("No additional parquet feature tables found; continuing with the base SPX feature set.")

# Remove degenerate zero-vol observations if present
if "target_ann_vol" in df5.columns:
    df5 = df5[df5["target_ann_vol"] != 0].reset_index(drop=True)

print(f"Final dataset size: {df5.shape[0]} rows × {df5.shape[1]} columns")
print(df5[["bar5", "target_ann_vol"]].head())

In [ ]:
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(exist_ok=True)
df5.to_parquet(OUTPUT_DIR / "train_features_5m_clean.parquet", index=False)
print(f"Saved: {OUTPUT_DIR / 'train_features_5m_clean.parquet'}")

In [14]:
# Sanity check of the generated data
df5.head()

,bar5,m1_open,m1_high,m1_low,m1_close,m2_open,m2_high,m2_low,m2_close,m3_open,...,prev_5m_log_return_3,prev_5m_log_return_4,prev_day_log_return,day_of_week,day_of_month,prev_5m_high_low_range,prev_atm_iv_30dte,prev_atm_iv_0dte,prev_total_option_volume,prev_total_option_oi
0,2025-02-05 09:55:00,6022.16,6022.72,6021.25,6022.17,6021.95,6022.58,6020.97,6021.46,6021.27,...,0.000158,0.000735,0.007199,2.0,5.0,8.38,0.131740,0.134747,116848.0,1676743.0
1,2025-02-05 10:00:00,6017.76,6021.79,6017.11,6020.79,6021.48,6030.16,6021.48,6025.86,6026.11,...,-0.000030,0.000158,0.007199,2.0,5.0,7.45,0.132913,0.132816,144866.0,1765872.0
2,2025-02-05 10:05:00,6010.62,6010.62,6007.06,6008.09,6007.96,6013.13,6007.19,6011.97,6011.77,...,-0.001316,-0.000030,0.007199,2.0,5.0,20.31,0.125245,0.193208,153967.0,2168613.0
3,2025-02-05 10:10:00,6011.87,6014.37,6011.87,6014.37,6014.51,6017.04,6014.51,6015.20,6015.23,...,-0.000645,-0.001316,0.007199,2.0,5.0,6.95,0.139310,0.107437,227213.0,2243066.0
4,2025-02-05 10:15:00,6016.03,6019.51,6015.51,6019.28,6019.39,6020.14,6018.68,6018.68,6018.20,...,-0.001245,-0.000645,0.007199,2.0,5.0,6.73,0.137197,0.113550,238342.0,2351741.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13802,2025-12-31 15:30:00,6854.12,6854.83,6853.10,6854.37,6854.36,6856.25,6852.95,6856.21,6856.34,...,0.000395,-0.000204,-0.001377,2.0,31.0,3.31,0.133368,0.048840,1255529.0,3449384.0
13803,2025-12-31 15:35:00,6859.43,6860.51,6859.14,6860.40,6860.37,6862.11,6860.03,6862.11,6862.08,...,-0.000787,0.000395,-0.001377,2.0,31.0,8.53,0.127231,0.020745,1681325.0,3058983.0
13804,2025-12-31 15:40:00,6860.30,6861.06,6859.84,6859.84,6859.74,6859.78,6858.14,6858.65,6858.60,...,-0.000315,-0.000787,-0.001377,2.0,31.0,5.04,0.130549,0.034435,1760851.0,3456363.0
13805,2025-12-31 15:45:00,6860.60,6861.68,6860.25,6861.49,6861.50,6861.96,6860.45,6860.45,6860.42,...,0.000764,-0.000315,-0.001377,2.0,31.0,2.92,0.128842,0.022609,1780742.0,3228600.0


In [ ]:
# Quick summary print
print(df5.describe(include="all").T.head(10))